In [ ]:
"""Prediksi PM2.5 multi-output dengan LSTM, GRU, dan Hybrid LSTM-GRU."""
from __future__ import annotations
import json, logging, pickle, random, sys
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import List, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
except ImportError as e:
    raise ImportError('Instal TensorFlow: python3 -m pip install tensorflow') from e

try: BASE_DIR=Path(__file__).resolve().parent
except NameError: BASE_DIR=Path.cwd()

@dataclass
class Config:
    data_file: Path = "tes2.csv"
    output_dir: Path = BASE_DIR/'OUTPUT_MODEL_PM25'
    time_col: str = 'waktu'
    target_col: str = 'PM2_5'
    feature_columns: List[str] = field(
    default_factory=lambda: [
        "PM10",
        "PM2_5",
        "humidity",
        "pressure",
        "solar_radiation",
        "temperature",
        "total_ch",
        "wind_direction",
        "wind_speed",
    ]
)
    freq: str='1min'
    lookback: int=60
    horizons: Tuple[int,...]=(10,30,60)
    train_ratio: float=.70
    val_ratio: float=.15
    interpolation_limit: int=10
    lstm_units: int=40
    gru_units: int=40
    dense_units: int=32
    dropout: float=.20
    lr: float=3e-4
    batch_size: int=128
    epochs: int=100
    patience: int=12
    seed: int=42
CFG=Config()

def logger(out):
    out.mkdir(parents=True,exist_ok=True)
    lg=logging.getLogger('pm25'); lg.setLevel(logging.INFO)
    if not lg.handlers:
        fmt=logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
        sh=logging.StreamHandler(sys.stdout); sh.setFormatter(fmt)
        fh=logging.FileHandler(out/'run.log',mode='w',encoding='utf-8'); fh.setFormatter(fmt)
        lg.addHandler(sh); lg.addHandler(fh)
    return lg

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); tf.keras.utils.set_random_seed(seed)

def load_data(c,lg):
    if not c.data_file.exists(): raise FileNotFoundError(c.data_file)
    df=pd.read_csv(c.data_file) if c.data_file.suffix.lower()=='.csv' else pd.read_excel(c.data_file)
    miss=[x for x in [c.time_col,*c.feature_cols] if x not in df.columns]
    if miss: raise KeyError(f'Kolom tidak ditemukan: {miss}')
    df[c.time_col]=pd.to_datetime(df[c.time_col],errors='coerce')
    df=df.dropna(subset=[c.time_col]).sort_values(c.time_col).drop_duplicates(c.time_col).set_index(c.time_col)
    for col in c.feature_cols: df[col]=pd.to_numeric(df[col],errors='coerce')
    df=df.reindex(pd.date_range(df.index.min(),df.index.max(),freq=c.freq))
    df.index.name=c.time_col
    qc={'humidity':(0,100),'pressure':(850,1100),'wind_speed':(0,75),'wind_direction':(0,360),
        'solar_radiation':(0,1500),'total_ch':(0,None),'PM1':(0,None),'PM2_5':(0,None),'PM10':(0,None)}
    for col,(lo,hi) in qc.items():
        if col in df:
            if lo is not None: df.loc[df[col]<lo,col]=np.nan
            if hi is not None: df.loc[df[col]>hi,col]=np.nan
    df[c.feature_cols]=df[c.feature_cols].interpolate('time',limit=c.interpolation_limit,limit_direction='both')
    before=len(df); df=df.dropna(subset=c.feature_cols)
    lg.info('Data bersih: %s; dibuang: %s',len(df),before-len(df))
    return df

def add_features(df,c):
    out=df.copy(); feats=[x for x in c.feature_cols if x!='wind_direction']
    if 'wind_direction' in out:
        rad=np.deg2rad(out['wind_direction']); out['wd_sin']=np.sin(rad); out['wd_cos']=np.cos(rad); feats+=['wd_sin','wd_cos']
    hour=out.index.hour; out['hour_sin']=np.sin(2*np.pi*hour/24); out['hour_cos']=np.cos(2*np.pi*hour/24); feats+=['hour_sin','hour_cos']
    return out,feats

def split(df,c):
    n=len(df); a=int(n*c.train_ratio); b=int(n*(c.train_ratio+c.val_ratio))
    return df.iloc[:a],df.iloc[a:b],df.iloc[b:]

def make_seq(df,feats,c,fs,ts):
    Xs=fs.transform(df[feats]).astype('float32')
    ys=ts.transform(df[[c.target_col]]).ravel().astype('float32')
    X=[]; y=[]; stamps=[]; hmax=max(c.horizons)
    for i in range(c.lookback,len(df)-hmax):
        X.append(Xs[i-c.lookback:i]); y.append([ys[i+h-1] for h in c.horizons]); stamps.append(df.index[i])
    return np.asarray(X,'float32'),np.asarray(y,'float32'),np.asarray(stamps)

def inv_multi(a,scaler):
    return np.column_stack([scaler.inverse_transform(a[:,i,None]).ravel() for i in range(a.shape[1])])

def compile_model(model,c):
    model.compile(optimizer=keras.optimizers.Adam(c.lr),loss=keras.losses.Huber(1.0),
                  metrics=[keras.metrics.MeanAbsoluteError(name='mae'),keras.metrics.RootMeanSquaredError(name='rmse')])
    return model

def build_model(kind,input_shape,nout,c):
    inp=keras.Input(shape=input_shape)
    if kind=='LSTM':
        x=layers.LSTM(c.lstm_units)(inp)
    elif kind=='GRU':
        x=layers.GRU(c.gru_units)(inp)
    else:
        x=layers.LSTM(c.lstm_units,return_sequences=True)(inp); x=layers.Dropout(c.dropout)(x); x=layers.GRU(c.gru_units)(x)
    x=layers.Dropout(c.dropout)(x); x=layers.Dense(c.dense_units,activation='relu')(x); out=layers.Dense(nout)(x)
    return compile_model(keras.Model(inp,out,name=kind),c)

def callbacks(folder,c):
    folder.mkdir(parents=True,exist_ok=True)
    return [keras.callbacks.EarlyStopping(monitor='val_loss',patience=c.patience,restore_best_weights=True),
            keras.callbacks.ModelCheckpoint(folder/'best_model.keras',monitor='val_loss',save_best_only=True),
            keras.callbacks.ReduceLROnPlateau(monitor='val_loss',factor=.5,patience=4,min_lr=1e-6),
            keras.callbacks.CSVLogger(folder/'history_log.csv')]

def metrics(name,y,p,horizons):
    rows=[]
    for i,h in enumerate(horizons):
        o=y[:,i]; q=p[:,i]
        rows.append({'model':name,'horizon_minutes':h,'R':np.corrcoef(o,q)[0,1],
                     'RMSE':np.sqrt(mean_squared_error(o,q)),'MAE':mean_absolute_error(o,q),
                     'R2':r2_score(o,q),'Bias':np.mean(q-o)})
    return pd.DataFrame(rows)

def run(c=CFG):
    lg=logger(c.output_dir); seed_all(c.seed)
    df=load_data(c,lg); df,feats=add_features(df,c); tr,va,te=split(df,c)
    fs=MinMaxScaler().fit(tr[feats]); ts=MinMaxScaler().fit(tr[[c.target_col]])
    Xtr,ytr,_=make_seq(tr,feats,c,fs,ts); Xv,yv,_=make_seq(va,feats,c,fs,ts); Xte,yte,tte=make_seq(te,feats,c,fs,ts)
    lg.info('Shapes train=%s val=%s test=%s',Xtr.shape,Xv.shape,Xte.shape)
    with open(c.output_dir/'feature_scaler.pkl','wb') as f: pickle.dump(fs,f)
    with open(c.output_dir/'target_scaler.pkl','wb') as f: pickle.dump(ts,f)
    allm=[]; preds={}; histories={}; ytrue=inv_multi(yte,ts)
    for kind in ['LSTM','GRU','HYBRID']:
        tf.keras.backend.clear_session(); folder=c.output_dir/kind
        model=build_model(kind,Xtr.shape[1:],len(c.horizons),c)
        hist=model.fit(Xtr,ytr,validation_data=(Xv,yv),epochs=c.epochs,batch_size=c.batch_size,
                       shuffle=False,callbacks=callbacks(folder,c),verbose=1)
        best=keras.models.load_model(folder/'best_model.keras')
        pred=inv_multi(best.predict(Xte,verbose=1),ts); preds[kind]=pred
        m=metrics(kind,ytrue,pred,c.horizons); allm.append(m)
        hdf=pd.DataFrame(hist.history); hdf.insert(0,'epoch',np.arange(1,len(hdf)+1)); histories[kind]=hdf
        hdf.to_csv(folder/'history.csv',index=False)
    result=pd.concat(allm,ignore_index=True).sort_values(['horizon_minutes','RMSE'])
    pred_df=pd.DataFrame({'waktu_prediksi':pd.to_datetime(tte)})
    for j,h in enumerate(c.horizons):
        pred_df[f'actual_{h}min']=ytrue[:,j]
        for k,v in preds.items(): pred_df[f'{k.lower()}_{h}min']=v[:,j]
    result.to_csv(c.output_dir/'model_metrics.csv',index=False); pred_df.to_csv(c.output_dir/'all_predictions.csv',index=False)
    with pd.ExcelWriter(c.output_dir/'REPORT_MODEL_PM25.xlsx',engine='openpyxl') as w:
        result.to_excel(w,'Metrics',index=False); pred_df.to_excel(w,'Predictions',index=False)
        for k,hdf in histories.items(): hdf.to_excel(w,f'History_{k}'[:31],index=False)
    for metric in ['RMSE','MAE','R','R2']:
        pivot=result.pivot(index='horizon_minutes',columns='model',values=metric)
        ax=pivot.plot(kind='bar',figsize=(10,6),title=f'Perbandingan {metric}'); ax.grid(axis='y',alpha=.3)
        plt.tight_layout(); plt.savefig(c.output_dir/f'comparison_{metric.lower()}.png',dpi=300); plt.close()
    (c.output_dir/'metadata.json').write_text(json.dumps({'features':feats,'config':{k:str(v) if isinstance(v,Path) else v for k,v in asdict(c).items()}},indent=2),encoding='utf-8')
    lg.info('Selesai: %s',c.output_dir.resolve()); return result

if __name__=='__main__':
    print(run().to_string(index=False))
